In [1]:
!pip install pandas numpy scikit-learn joblib datasets great_expectations optuna wandb gradio evidently
!pip install transformers torch gradio
!pip install -q great_expectations

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.0/238.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 579.2/579.2 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 41.3 MB/s eta 0:00:00


In [2]:
import uuid
import pandas as pd
import great_expectations as gx
from transformers import pipeline
import gradio as gr
import optuna
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras import layers, models


**ACCESO AL DATASET de HUGGING FACE**

In [3]:

from datasets import load_dataset
dataset = load_dataset("rotten_tomatoes")

train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

train_df.head()

The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

,text,label
0,the rock is destined to be the 21st century's ...,1
1,"the gorgeously elaborate continuation of "" the...",1
2,effective but too-tepid biopic,1
3,if you sometimes like to go to the movies to h...,1
4,"emerges as something rare , an issue movie tha...",1


**FUNCIÓN PARA TRATAR LAS "EXPECTATIVAS"**

In [10]:
import uuid
import pandas as pd
import great_expectations as gx

def validate_rotten_tomatoes_df(df, name="dataset"):
  context = gx.get_context()

  data_source = context.sources.add_pandas(name=name)

  rotten_tomatoes_dataset = data_source.add_dataframe(dataframe=df)

  batch_definition = rotten_tomatoes_dataset.get_batch_definition_whole_dataframe()

  batch = rotten_tomatoes_dataset.get_batch(batch_kwargs={"data_asset_name": name})

  expectations = [
      gx.expectations.ExpectTableColumnsToMatchSet(
          column_set=["text", "label"],
          exact_match=True
      ),
      gx.exceptions.ExceptColumnValuesToNottBeNull(
          column="text"
      )
  ]

  results = []
  for expectation in expectations:
      result = expectation.validate(batch)
      results.append({
          "dataset": name,
          "sucess": result.sucess
          })

  return results

**VALIDAMOS EL DF**

In [11]:
train_validation = validate_rotten_tomatoes_df(train_df)
test_validation = validate_rotten_tomatoes_df(test_df)

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmp1wl3fl7v' for ephemeral docs site


AttributeError: 'EphemeralDataContext' object has no attribute 'sources'

**DESCARGAMOS EL MODELO DE HF**

**INTERFAZ EN GRADIO**

**USO DE OPTUNA**

**VISUALIZACIÓN DE OPTUNA**